# Estación más cercana, en cualquier país -- sin anclar nada a San José (ECO | Wind)

Corrección de Pablo sobre todo el trabajo de Hallazgo 17-22: ECO | Wind es una app
**internacional**, no debería estar comparando o prestando forma climática de San José para
resolver un sitio nuevo. Con el catálogo global ya en el repo (`datos_clima/epw_catalog_global.json`,
Hallazgo 19 -- el mismo que usan Skyplus/DDP-lite) hay **5,276 estaciones reales en 20 países**
(USA 2,969, Canadá 914, Brasil 667, México 173, Argentina 116, Chile 69, Colombia 46, Ecuador 45,
Perú 39, Bolivia 39, Venezuela 37, Paraguay 29, Panamá 28, Uruguay 21, Rep. Dominicana 19,
Honduras 18, Guatemala 17, Costa Rica 12, Nicaragua 11, El Salvador 7) -- Costa Rica es apenas el
0.2% del catálogo. Para casi cualquier punto de interés hay una estación real y cercana: no hace
falta "pedir prestada" una forma de otro sitio, se puede usar el dato real de la estación más
cercana directo.

**Lo que prueba este notebook:** el mismo algoritmo (`obtener_estaciones_cercanas()` +
`descargar_y_extraer_epw()`, ya en `engine/epw_real.py`, mismo patrón que Skyplus/DDP-lite) tiene
que poder "auto-pivotar" a cualquier punto dentro del catálogo -- se prueba con puntos reales en
varios países distintos, no sólo Costa Rica -- y una vez que baja la estación real más cercana, la
"sensibiliza" con lo que el cliente quiere saber: producción de energía real (kWh/año) para su
configuración de turbinas, corriendo el mismo motor ya validado (`simular()`, Hallazgo 12/17/20).

**Ningún punto de este notebook usa San José como referencia de forma ni de comparación** -- cada
punto se resuelve solo, con su propia estación real más cercana. Los 4 archivos "base" (San José,
Nicoya, Liberia, Finca Favorita) no se tratan distinto de cualquier otro punto -- si el algoritmo
encuentra que la estación más cercana a uno de ellos es una de esas 4, es sólo coincidencia
geográfica, no un caso especial en el código.

**No corre completo en el sandbox de desarrollo** (climate.onebuilding.org bloqueado, Hallazgo 2)
-- la búsqueda de la estación más cercana SÍ corre (usa el catálogo local + un fallback sin red
para inferir el país, ver `obtener_estaciones_cercanas()`), pero la descarga real del EPW necesita
Colab.

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at 68d6d7e feat(fase2): notebook para descargar las 8 estaciones de Costa Rica que faltan


/home/user/eco-wind/notebooks
Commit activo: 68d6d7e  feat(fase2): notebook para descargar las 8 estaciones de Costa Rica que faltan  (2026-08-31 21:23:55 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from engine.epw_real import obtener_estaciones_cercanas, descargar_y_extraer_epw, cargar_epw_real
from engine.simulador_pista_a import simular

pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

## Puntos de prueba -- varios países, no sólo Costa Rica

Coordenadas reales de ciudades/proyectos en 6 países distintos del catálogo, para probar que el
algoritmo de verdad "auto-pivota" -- ninguna de estas es necesariamente la coordenada exacta de una
estación (son ciudades reales; la búsqueda tiene que encontrar la estación real más cercana a cada
una, sea cual sea).

In [3]:
PUNTOS_PRUEBA = [
    dict(nombre="San José, Costa Rica",        lat=9.9281,   lon=-84.0907),
    dict(nombre="Bogotá, Colombia",             lat=4.7110,   lon=-74.0721),
    dict(nombre="Ciudad de México, México",     lat=19.4326,  lon=-99.1332),
    dict(nombre="Buenos Aires, Argentina",      lat=-34.6037, lon=-58.3816),
    dict(nombre="São Paulo, Brasil",            lat=-23.5505, lon=-46.6333),
    dict(nombre="Ciudad de Panamá, Panamá",     lat=8.9824,   lon=-79.5199),
]
for p in PUNTOS_PRUEBA:
    print(f"  {p['nombre']:30s} ({p['lat']:.4f}, {p['lon']:.4f})")

  San José, Costa Rica           (9.9281, -84.0907)
  Bogotá, Colombia               (4.7110, -74.0721)
  Ciudad de México, México       (19.4326, -99.1332)
  Buenos Aires, Argentina        (-34.6037, -58.3816)
  São Paulo, Brasil              (-23.5505, -46.6333)
  Ciudad de Panamá, Panamá       (8.9824, -79.5199)


## Una sola función, sin ningún caso especial: estación real más cercana → producción del cliente

`evaluar_punto()` hace exactamente lo que pidió Pablo: para cualquier (lat, lon), busca la
estación real más cercana en el catálogo global, descarga su EPW real, y corre el motor de
simulación ya validado para la configuración de turbinas que el cliente quiera evaluar. No hay
ninguna referencia a San José, ni ninguna forma "prestada" -- es la propia estación real la que se
usa, siempre.

In [4]:
def evaluar_punto(lat, lon, modelo="medium_tulip", N=3, altura_buje=3.0):
    cercanas = obtener_estaciones_cercanas(lat, lon, top_n=1)
    if cercanas is None or cercanas.empty:
        return dict(error="No se encontró ninguna estación real cerca de este punto en el catálogo.")

    estacion = cercanas.iloc[0]
    try:
        ruta_epw = descargar_y_extraer_epw(estacion["url"])
        df_clima, meta = cargar_epw_real(ruta_epw)
    except Exception as exc:
        return dict(
            estacion_real=estacion["name"], pais=estacion.get("country", estacion.get("state", "?")),
            distancia_km=estacion["distancia_km"],
            error=f"No se pudo descargar/leer el EPW real: {exc!r}",
        )

    r = simular(df_clima, altura_buje=altura_buje, modelo=modelo, N=N, elevacion_m=meta["elevacion_m"])
    return dict(
        estacion_real=meta["estacion"], pais=meta["pais"], wmo=meta["wmo"],
        distancia_km=round(float(estacion["distancia_km"]), 1),
        lat_estacion=meta["lat"], lon_estacion=meta["lon"], elevacion_m=meta["elevacion_m"],
        media_real_m_s=round(float(df_clima["WS10M"].mean()), 3),
        kwh_anual=round(r["kwh_anual"], 1), v_hub_medio=round(r["v_hub_medio"], 3),
        error=None,
    )

In [5]:
resultados = []
for p in PUNTOS_PRUEBA:
    print(f"=== {p['nombre']} ===")
    r = evaluar_punto(p["lat"], p["lon"])
    if r.get("error"):
        print(f"  {r['error']}")
    else:
        print(f"  Estación real usada: {r['estacion_real']} ({r['pais']}) -- {r['distancia_km']} km del punto")
        print(f"  Media real: {r['media_real_m_s']} m/s | Producción (medium_tulip x3, buje 3m): "
              f"{r['kwh_anual']:,.0f} kWh/año")
    resultados.append({"punto_consultado": p["nombre"], **r})
    print()

pd.DataFrame(resultados)

=== San José, Costa Rica ===


  No se pudo descargar/leer el EPW real: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_SJ_San.Jose-La.Sabana.787605_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Bogotá, Colombia ===


  No se pudo descargar/leer el EPW real: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_3_South_America/COL_Colombia/MET_Meta/COL_MET_Las.Gaviotas.802410_TMYx.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Ciudad de México, México ===


  No se pudo descargar/leer el EPW real: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/MEX_Mexico/MEX_Mexico/MEX_MEX_Fes.Cuautitlan.766700_TMYx.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Buenos Aires, Argentina ===


  No se pudo descargar/leer el EPW real: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_3_South_America/ARG_Argentina/BA_Buenos_Aires/ARG_BA_El.Palomar.AP.875710_TMYx.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== São Paulo, Brasil ===


  No se pudo descargar/leer el EPW real: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_3_South_America/BRA_Brazil/SP_Sao_Paulo/BRA_SP_Sao.Paulo.869100_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Ciudad de Panamá, Panamá ===


  No se pudo descargar/leer el EPW real: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/PAN_Panama/PAN_CL_Ft.Sherman.Rocob.788010_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))



,punto_consultado,estacion_real,pais,distancia_km,error
0,"San José, Costa Rica",San Jose La Sabana,CRI,2.10,No se pudo descargar/leer el EPW real: ProxyEr...
1,"Bogotá, Colombia",Las Gaviotas,COL,84.65,No se pudo descargar/leer el EPW real: ProxyEr...
2,"Ciudad de México, México",Fes Cuautitlan,MEX,29.63,No se pudo descargar/leer el EPW real: ProxyEr...
3,"Buenos Aires, Argentina",El Palomar AP,ARG,19.70,No se pudo descargar/leer el EPW real: ProxyEr...
4,"São Paulo, Brasil",Sao Paulo,BRA,0.02,No se pudo descargar/leer el EPW real: ProxyEr...
5,"Ciudad de Panamá, Panamá",Ft Sherman Rocob,PAN,23.34,No se pudo descargar/leer el EPW real: ProxyEr...


## Conclusión

Si las celdas de arriba corrieron con éxito en Colab, cada fila de la tabla final salió con su
**propia** estación real -- sin ningún punto anclado a San José, sin ninguna forma prestada. Eso es
justo lo que hace falta para una app internacional: el algoritmo se auto-pivota a la estación real
más cercana dentro de cualquiera de los 20 países del catálogo, y de ahí en adelante corre el mismo
motor de simulación ya validado (Hallazgo 12/17/20) para darle al cliente lo que quiere saber
(producción de energía real para su proyecto).

**Qué falta para llevar esto a `app.py`:** el camino principal de la app (buscar → elegir estación
→ simular) YA funciona así -- no hace falta escribirlo de nuevo. Lo que si convendría reconsiderar,
con este resultado en la mano, es si vale la pena mantener la aproximación de respaldo que pide
prestada la forma de San José (`cargar_aproximacion()`, `engine/gwa_raster.py`) para cuando no hay
estación real cerca: con 5,276 estaciones en 20 países, ese caso debería ser mucho más raro de lo
que parecía cuando sólo se probó con Costa Rica. Decisión de producto para Pablo, no tomada acá.